# Lesson 8: Forward modeling and dipoles

Neurocampus course "Signals of the whole brain"

Daria Kleeva

dkleeva@gmail.com

May 6th, 2026


## FreeSurfer

In [ ]:
# export FREESURFER_HOME=/Applications/freesurfer
# export SUBJECTS_DIR=$FREESURFER_HOME/subjects
# source $FREESURFER_HOME/SetUpFreeSurfer.sh 
# mri_convert "/Users/dkleeva/Desktop/SubjectName/SubjectName_MRI/IMG-0003-00001.dcm" SUBJECTS_DIR/SubjectName.nii.gz
# recon-all -i $SUBJECTS_DIR/SubjectName.nii.gz -subjid SubjectName
# recon-all -all -subjid SubjectName -parallel -openmp 8

In [ ]:
# import mne
# from mne.bem import make_watershed_bem
# fs='/Applications/freesurfer/subjects/'
# mne.bem.make_watershed_bem('SubjectName', subjects_dir=fs, overwrite=True, volume='T1', 
#                            atlas=False, gcaatlas=False, preflood=None, show=False, copy=False, T1=None, brainmask='ws.mgz', verbose=None)

## Import

In [ ]:
import mne
from mne import io
import matplotlib.pyplot as plt
import numpy as np

from mne.simulation import simulate_raw, add_noise
from mne.datasets import sample
import seaborn as sns

In [ ]:
from mne.datasets import sample
data_path = sample.data_path()
raw_fname = str(data_path) + '/MEG/sample/sample_audvis_raw.fif'
event_fname = str(data_path) + '/MEG/sample/sample_audvis_raw-eve.fif'

In [ ]:
raw = mne.io.read_raw_fif(raw_fname, preload=True)
events = mne.read_events(event_fname)

In [ ]:
import os.path as op
subjects_dir = op.join(data_path, 'subjects')

## Spherical head model

In [ ]:
meg_info = raw.copy().pick('mag').info
sphere = mne.make_sphere_model(
    r0='auto',          
    head_radius='auto',  
    info=meg_info,
)

In [ ]:
r0 = sphere['r0']

In [ ]:
true_pos_rel = np.array([0.028, -0.045, -0.010])  # meters, offset from r0
true_pos = (r0 + true_pos_rel).reshape(1, 3)

In [ ]:
distance_from_center = np.linalg.norm(true_pos_rel) * 1000
print(f"Position of dipole: {true_pos[0]*1000} mm")
print(f"Distance from center: {distance_from_center:.1f} mm")
print(f"(brain radius in our model: {sphere['layers'][0]['rad']*1000:.1f} mm)")

In [ ]:
def make_tangent(pos_rel, template):
    """Make a tangent vector to the sphere at a given point."""
    radial = pos_rel / np.linalg.norm(pos_rel)
    template = np.asarray(template, dtype=float)
    tangent = template - np.dot(template, radial) * radial
    return tangent / np.linalg.norm(tangent)

In [ ]:
true_ori = make_tangent(true_pos_rel, [0, 1, 0]).reshape(1, 3)
print(f"\nOrientation of dipole: {true_ori[0]}")

In [ ]:
true_amp = 50e-9  # A·m
true_dipole = mne.Dipole(
    times=np.array([0.0]),                    # one moment in time
    pos=true_pos,                            # position
    amplitude=np.array([true_amp]),          # amplitude
    ori=true_ori,                            # orientation
    gof=np.array([100.0]),                   # GOF — now not important
)

In [ ]:
fwd_one, _ = mne.make_forward_dipole(
    dipole=true_dipole,
    bem=sphere,
    info=meg_info,
    trans=None,   # for a spherical model, coordinates coincide
)

In [ ]:
G_one = fwd_one['sol']['data']

In [ ]:
m_meg = (G_one * true_amp).ravel()   # (n_channels,) in volts
 

m_meg_fmt = m_meg * 1e6
print(f"Signal on MEG sensors:")
print(f"  range: [{m_meg_fmt.min():.2f}, {m_meg_fmt.max():.2f}] μV")
print(f"  mean:  {m_meg_fmt.mean():.2f} μV")
print(f"  RMS:      {np.sqrt(np.mean(m_meg_fmt**2)):.2f} μV")
 

fig, ax = plt.subplots(figsize=(5, 5))
mne.viz.plot_topomap(
    m_meg_fmt, meg_info, axes=ax, show=False,
    cmap='RdBu_r', sensors=True, contours=6,
    sphere=sphere,
)
ax.set_title(f'Topography from a dipole in the right occipital area\n'
             f'(orientation — tangential, amplitude = {true_amp*1e9:.0f} nA·m)',
             fontsize=11)
plt.tight_layout()
plt.show()

## Dipoles parameters

In [ ]:
def topo_from_dipole(pos_rel, ori_template, amp, info, sphere):
    pos_rel = np.asarray(pos_rel, dtype=float)
    pos = (sphere['r0'] + pos_rel).reshape(1, 3)
    ori = make_tangent(pos_rel, ori_template).reshape(1, 3)
    dip = mne.Dipole(
        times=np.array([0.0]), pos=pos,
        amplitude=np.array([amp]),
        ori=ori, gof=np.array([100.0]),
    )
    fwd, _ = mne.make_forward_dipole(dip, sphere, info)
    topo = (fwd['sol']['data'] * amp).ravel()
    return topo, pos, ori

In [ ]:
x_positions_mm = np.linspace(-40, 40, 5)   
 
fig, axes = plt.subplots(1, len(x_positions_mm), figsize=(15, 3.5))
 
all_topos_x = []
for x_mm in x_positions_mm:
    pos_rel = np.array([x_mm/1000, -0.045, -0.010])
    topo, _, _ = topo_from_dipole(pos_rel, [0, 1, 0], 50e-9, meg_info, sphere)
    all_topos_x.append(topo)
vmax = np.max(np.abs(all_topos_x)) * 1e6
 
for ax, topo, x_mm in zip(axes, all_topos_x, x_positions_mm):
    mne.viz.plot_topomap(
        topo * 1e6, meg_info, axes=ax, show=False,
        cmap='RdBu_r', vlim=(-vmax, vmax), sphere=sphere,
        sensors=True, contours=4,
    )
    ax.set_title(f'x = {x_mm:+.0f} мм', fontsize=11)
 
fig.suptitle('Effect of position: shift of dipole horizontally', y=1.05, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
distances_mm = [20, 40, 55, 70, 80]   
brain_radius_mm = sphere['layers'][0]['rad'] * 1000
 
fig, axes = plt.subplots(1, len(distances_mm), figsize=(15, 3.5))
 
all_topos_depth = []
for d_mm in distances_mm:
    direction = np.array([0., -0.5, -0.3])
    direction /= np.linalg.norm(direction)
    pos_rel = direction * d_mm / 1000
    topo, _, _ = topo_from_dipole(pos_rel, [1, 0, 0], 50e-9, meg_info, sphere)
    all_topos_depth.append(topo)
 
for ax, topo, d_mm in zip(axes, all_topos_depth, distances_mm):
    depth_from_surface = brain_radius_mm - d_mm
    vmax_local = np.max(np.abs(topo)) * 1e6
    mne.viz.plot_topomap(
        topo * 1e6, meg_info, axes=ax, show=False,
        cmap='RdBu_r', vlim=(-vmax_local, vmax_local),
        sphere=sphere, sensors=True, contours=4,
    )
    peak = np.max(np.abs(topo)) * 1e6
    focality = (np.max(topo**2) / np.sum(topo**2)) * len(topo)
    ax.set_title(f'глубина {depth_from_surface:.0f} мм\n'
                 f'пик={peak:.2f} мкВ\n'
                 f'focality={focality:.2f}', fontsize=9)
 
fig.suptitle('Effect of depth: deep source is «spread» over the entire scalp,\n'
             'surface source is concentrated',
             y=1.10, fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
ori_templates = [
    ('tangential forward', [0, 1, 0]),
    ('tangential right', [1, 0, 0]),
    ('tangential up',  [0, 0, 1]),
]
 
pos_rel_fixed = np.array([0.028, -0.045, -0.010])
 
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
all_topos_ori = []
for label, tmpl in ori_templates:
    topo, _, _ = topo_from_dipole(pos_rel_fixed, tmpl, 50e-9, meg_info, sphere)
    all_topos_ori.append(topo)
 
vmax_ori = np.max(np.abs(all_topos_ori)) * 1e6
 
for ax, (label, _), topo in zip(axes, ori_templates, all_topos_ori):
    mne.viz.plot_topomap(
        topo * 1e6, meg_info, axes=ax, show=False,
        cmap='RdBu_r', vlim=(-vmax_ori, vmax_ori),
        sphere=sphere, sensors=True, contours=5,
    )
    ax.set_title(label, fontsize=11)
 
fig.suptitle('Effect of orientation: one dipole, three different directions',
             y=1.05, fontsize=13)
plt.tight_layout()
plt.show()

## Compute BEM model and solution

In [ ]:
src = mne.setup_source_space('sample', spacing='ico4',
                                 subjects_dir=subjects_dir)
model = mne.make_bem_model(subject='sample', ico=4, conductivity=[0.3, 0.006, 0.3],
                               subjects_dir=subjects_dir)
bem = mne.make_bem_solution(model)

In [ ]:
fig = mne.viz.plot_bem(subject='sample', subjects_dir=subjects_dir,
                 brain_surfaces='white', orientation='coronal', src=src)

In [ ]:
src

In [ ]:
src[0]

In [ ]:
src[0]['rr']

In [ ]:
src[0]['vertno']

In [ ]:
src[0]['nn']

In [ ]:
plt.scatter(src[0]['rr'][:,1], src[0]['rr'][:,2])
plt.show()

In [ ]:
plt.scatter(src[1]['rr'][:,1], src[1]['rr'][:,2])
plt.show()

In [ ]:
src[0]['nn']

## Coregistration

In [ ]:
mne.gui.coregistration(subject='sample', subjects_dir=subjects_dir)

In [ ]:
trans = str(data_path) + '/MEG/sample/sample_audvis_raw-trans.fif'

In [ ]:
info = mne.io.read_info(raw_fname)
mne.viz.plot_alignment(info, trans, subject='sample', dig=True,
                       meg=['helmet', 'sensors'], subjects_dir=subjects_dir,
                       surfaces='head-dense')

## Compute forward solution

In [ ]:
fwd = mne.make_forward_solution(raw_fname, trans=trans, src=src, bem=bem,
                                meg=True, eeg=True, mindist=5.0, n_jobs=1,
                                verbose=True)

In [ ]:
leadfield = fwd['sol']['data']

In [ ]:
leadfield.shape

In [ ]:
leadfield.shape[1]/3

In [ ]:
fwd

In [ ]:
fwd_fixed = mne.convert_forward_solution(fwd, surf_ori=True, force_fixed=True,
                                         use_cps=True)
leadfield_fixed = fwd_fixed['sol']['data']

In [ ]:
leadfield_fixed

In [ ]:
%matplotlib inline

In [ ]:
ind_show = 50
one_chan_leadfield = leadfield_fixed[ind_show]
vert_used = fwd_fixed['src'][0]['vertno']
fwd_to_plot = fwd_fixed['src'][0]['rr'][vert_used,:]
plt.scatter(fwd_to_plot[:,1],fwd_to_plot[:,2], c = one_chan_leadfield[:int(fwd_to_plot.shape[0])], cmap = 'plasma')

In [ ]:
raw.info['ch_names'][ind_show]

In [ ]:
fig = raw.plot_sensors(show_names=True, ch_type='mag')

## Plot topographies

In [ ]:
column=300
ind = mne.pick_types(raw.info, meg='mag',eeg=False)
info=mne.pick_info(raw.info, ind)
data=leadfield[ind,:]
fig=mne.viz.plot_topomap(data[:,column], pos = info)

In [ ]:
column=301
ind = mne.pick_types(raw.info, meg='mag',eeg=False)
info=mne.pick_info(raw.info, ind)
data=leadfield[ind,:]
fig=mne.viz.plot_topomap(data[:,column], pos = info)

In [ ]:
column=302
ind = mne.pick_types(raw.info, meg='mag',eeg=False)
info=mne.pick_info(raw.info, ind)
data=leadfield[ind,:]
fig=mne.viz.plot_topomap(data[:,column], pos = info)

## Dipole modeling

Interactive dipole simulator: https://mybinder.org/v2/gh/mne-tools/dipole-simulator/master?urlpath=voila%2Frender%2Findex.ipynb

In [ ]:
amplitude = np.array([1]).reshape(1,)  
ori = np.array([1.,1.,1.]).reshape(1, 3)  
ori /= np.linalg.norm(ori)
pos=np.array([0.003, -0.01, 0.1])
pos = pos.reshape(1, 3)
gof = np.array([100]).reshape(1,)
dip = mne.Dipole(times=[0], pos=pos, ori=ori,
                     amplitude=amplitude, gof=gof)
fwd, _ = mne.make_forward_dipole(dip, bem=bem, info=info, trans=trans,
                                     verbose=True)
fwd = mne.convert_forward_solution(fwd, force_fixed=False, verbose=True)

In [ ]:
dip

In [ ]:
fig=dip.plot_locations(trans, 'sample', subjects_dir, mode='orthoview')


In [ ]:
leadfield_dip = fwd['sol']['data']

In [ ]:
column=0
ind = mne.pick_types(raw.info, meg='mag',eeg=False)
info=mne.pick_info(raw.info, ind)
data=leadfield_dip
fig=mne.viz.plot_topomap(data[:,column], pos = info)

In [ ]:
dipole_ori = ori.reshape(3, 1)
lf_fixed = leadfield_dip @ dipole_ori

In [ ]:
ind = mne.pick_types(raw.info, meg='mag',eeg=False)
info=mne.pick_info(raw.info, ind)
fig=mne.viz.plot_topomap(np.reshape(lf_fixed, [lf_fixed.shape[0],]), pos = info)